In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import os
import csv
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import shutil
import time
from torch.utils.data import DataLoader, Dataset, random_split,Subset
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
from torchsummary import summary
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from pathlib import Path
from torchsampler import ImbalancedDatasetSampler
from sklearn.preprocessing import LabelEncoder

In [2]:
torch.cuda.is_available()

True

In [3]:
transform=transforms.Compose([
    transforms.Resize((128,128)),     #將圖片大小縮放至(128*128)
    transforms.ToTensor()
])

In [4]:
#建立dataset class
class ImageDataset(Dataset):
    def __init__(self, root,transform,decision_mode=False):
        super().__init__()
        self.images = ImageFolder(root=root, transform=transform)
        self.classes = self.images.classes
        self.flag= [False] * len(self.images)


        self.decision_mode=decision_mode

        
    def __len__(self):
        return len(self.images)
   
    def __getitem__(self,idx):
        if(self.decision_mode==False):
            image, label=self.images[idx]
            
            return image, label,idx
        else:
            image, _=self.images[idx]

            label=int(self.flag[idx])
            return image,label,idx
    
    def update_flag(self, idx):
        self.flag[idx] = True
    

In [5]:
#讀取資料且切分資料為train、valid、test
DATA_DIR = r'C:\Users\MJ\Desktop\fish\Fish_Dataset\Fish_Dataset'
images=ImageDataset(DATA_DIR,transform,decision_mode=False)

size = len(images)
valid_size = int(0.2 * size)
test_size = int(0.2 * size)
train_size = int(size - valid_size - test_size)

trainset, validset, testset = random_split(images, (train_size, valid_size, test_size))


In [6]:
# create data loaders
batch_size = 64 # larger numbers lead to CUDA running out of memory
train_dl = DataLoader(trainset,shuffle=False, batch_size=batch_size)
valid_dl = DataLoader(validset,shuffle=False, batch_size=batch_size)
test_dl = DataLoader(testset, batch_size=batch_size)
criterion = nn.CrossEntropyLoss()

In [7]:
#建立模型框架
def model_create(model_name):
    model = getattr(models,model_name)(weights=True)
    if(model_name=='googlenet' or model_name=='resnet18' ):
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, len(trainset.dataset.classes))
    else :
        model.classifier._modules['6'] = nn.Sequential(nn.Linear(4096,len(trainset.dataset.classes)))
    model.cuda()
    return model

In [8]:
model_list={'alexnet':'alexnet',
        'vgg19':'vgg19',
        'resnet18':'resnet18',
        'googlenet':'googlenet'
        }

In [9]:
#訓練模型
def trainer(epochs,model,criterion,optim,data_dl):
        for epoch in range(epochs):
                model.train()
                ###################
                # train the model #
                ###################
                for data, target,idx in data_dl:
                        
                        optim.zero_grad()
                        data,target=data.cuda(),target.cuda()  #將data、target放到gpu上

                        out = model(data)
                        _, y_pred_tag = torch.max(out, dim = 1)  

                        loss = criterion(out, target)
                        loss.backward()
                        
                        optim.step()
        return model
          

In [10]:
#將模型建立框架後並訓練
def model_train(model_name,data_dl,data_name,epochs):
    model=model_create(model_name)
    criterion = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    if(data_name=="decision"):
        trainset.dataset.decision_mode=True
        model=trainer(epochs,model,criterion,optim,data_dl)
        trainset.dataset.decision_mode=False
    else:
        model=trainer(epochs,model,criterion,optim,data_dl)
    torch.save(model, f"model_{model_name}_{data_name}.pth")
    return model

In [11]:
# model_list['googlenet']=model_train('googlenet',train_dl,"original",5)

In [12]:
# for model_name in model_list:
#     model_list[model_name]=model_train(model_name,train_dl,"original",50)
    

In [13]:
#讀取所有模型
for model_name in model_list:
    model_list[model_name] = torch.load(f'C:/Users/MJ/Desktop/mjthesis/model_{model_name}.pth')

In [14]:
#將資料切分成true and false
def split_data(model_name):
    model_list[model_name].eval()

    indexF=[]
    indexT=[]
    with torch.no_grad():
        for (data,target,idx) in train_dl:
            model_list[model_name] = model_list[model_name].to(torch.device('cpu'))
            out = model_list[model_name](data)
            _, y_pred_tag = torch.max(out, dim = 1) 

            for idx,result in zip(idx,torch.eq(target,y_pred_tag)):

                if(result.cpu().numpy()):
                    indexT.append(idx.cpu().numpy().item())
                    trainset.dataset.update_flag(idx)
                else:
                    indexF.append(idx.cpu().numpy().item())
                    
    return indexF,indexT

In [21]:
#將資料切分後建立dataset與dataloader
model_name='googlenet'
indexF=[]
indexT=[]
indexF,indexT=split_data(model_name)

Fdataset=Subset(trainset.dataset, indexF)
Tdataset=Subset(trainset.dataset, indexT)
Fdl=DataLoader(Fdataset, shuffle=True, batch_size=int(len(indexF)/2))
Tdl=DataLoader(Tdataset, shuffle=True, batch_size=int(len(indexF)/2))

decision_dl = DataLoader(trainset, batch_size=batch_size)


In [18]:
torch.cuda.empty_cache()

In [22]:
#訓練模型T、F、decision
# model_T=model_train(model_name,Tdl,"T",10)
model_T=torch.load(f'C:/Users/MJ/Desktop/mjthesis/model_{model_name}_T.pth')

In [41]:
# model_F=model_train(model_name,Fdl,"F",10)
model_F=torch.load(f'C:/Users/MJ/Desktop/mjthesis/model_{model_name}_F.pth')

c:\Users\MJ\anaconda3\envs\MJ\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [24]:
# model_decision=model_train(model_name,decision_dl,"decision",20)
model_decision = torch.load(f'C:/Users/MJ/Desktop/mjthesis/model_{model_name}_decision.pth')

In [25]:
#模型表現
def evaluate_model(model,data_dl,size):
        model.cuda()
        ######################    
        # validate the model #
        ######################
        total_loss = 0
        accu = 0
        with torch.no_grad():
                model.eval()   

                for data, target,idx in data_dl:

                        data,target=data.cuda(),target.cuda()
                        out= model(data)
                        _, y_pred_tag = torch.max(out, dim = 1)
                        loss = criterion(out, target)

                        total_loss+= loss.item()*data.size(0)
                        correct=torch.sum(y_pred_tag == target).item()
                        accu += correct      

                total_loss=total_loss/size
                accu=accu/size
        return total_loss,accu

In [26]:
train_loss,train_accu=evaluate_model(model_list[model_name],train_dl,train_size)

In [27]:
train_loss,train_accu

(0.015818091508178138, 0.9962037037037037)

In [37]:
valid_loss,valid_accu=evaluate_model(model_list[model_name],valid_dl,valid_size)

In [38]:
valid_loss,valid_accu

(0.01418498268965171, 0.9961111111111111)

In [39]:
test_loss,test_accu=evaluate_model(model_list[model_name],test_dl,test_size)

In [40]:
test_loss,test_accu

(0.02401318583183133, 0.995)

In [28]:
trainset.dataset.decision_mode=True
decision_loss,decision_accu=evaluate_model(model_decision,decision_dl,train_size)
trainset.dataset.decision_mode=False

In [29]:
decision_loss,decision_accu

(0.0209410857476501, 0.9965740740740741)

In [30]:
T_loss,T_accu=evaluate_model(model_T,Tdl,len(Tdataset))

In [31]:
T_loss,T_accu

(0.1648678991967662, 0.9459057533227995)

In [43]:
F_loss,F_accu=evaluate_model(model_F,Fdl,len(Fdataset))

In [44]:
F_loss,F_accu

(0.21049154799704145, 0.926829268292683)

In [45]:
#全部模型裝在一起的表現
def total_model_evaluate(data_dl,size):
    total_loss = 0
    accu = 0
    with torch.no_grad():
        for data,target,idx in data_dl:
            outputs=[]
            data,target=data.cuda(),target.cuda()
            out = model_decision(data)
            _, y_pred_tag = torch.max(out, dim = 1)
            
            sum=0
            for pred in y_pred_tag:
                if(pred==0):
                    outputs=model_F(data)

                else:
                    outputs=model_T(data)

            _, y_pred_tag = torch.max(outputs, dim = 1)
            loss = criterion(outputs, target)
            
            
            total_loss+= loss.item()*data.size(0)
            correct=torch.sum(y_pred_tag == target).item()
            accu += correct      

    total_loss=total_loss/size
    accu=accu/size
    return total_loss,accu
        
        

In [46]:
total_loss,total_accu=total_model_evaluate(valid_dl,valid_size)

In [48]:
total_loss,total_accu

(0.1564330607983801, 0.9402777777777778)